# Nhận diện biển báo giao thông bằng YOLO11

Quy trình tải dữ liệu, tiền xử lý, huấn luyện mô hình YOLO11 phiên bản Nano và kiểm tra kết quả dự đoán thực tế.

In [ ]:
import os
import kagglehub

maitam_vietnamese_traffic_signs_path = kagglehub.dataset_download('maitam/vietnamese-traffic-signs')

print('Downloaded dataset path:', maitam_vietnamese_traffic_signs_path)
if os.path.exists(maitam_vietnamese_traffic_signs_path):
    print('Contents of the downloaded directory:')
    for item in os.listdir(maitam_vietnamese_traffic_signs_path):
        item_path = os.path.join(maitam_vietnamese_traffic_signs_path, item)
        if os.path.isdir(item_path):
            print(f' [DIR]  {item}/ -> contains: {os.listdir(item_path)[:5]}')
        else:
            print(f' [FILE] {item}')
else:
    print('The path does not exist!')

In [ ]:
import os

# Đường dẫn gốc do kagglehub trả về 
base_path = maitam_vietnamese_traffic_signs_path

def find_dataset_root(base):
    """Tự dò thư mục chứa cả 'images' và 'labels'"""
    for root, dirs, files in os.walk(base):
        if 'images' in dirs and 'labels' in dirs:
            return root
    return None

src_folder = find_dataset_root(base_path)
assert src_folder is not None, (
    f"Không tìm thấy thư mục chứa 'images' và 'labels' trong {base_path}. "
    "Hãy xem lại cấu trúc ở cell kế tiếp."
)

dst_folder = '/kaggle/working/dataset_yolov11'
project_dir = '/kaggle/working/runs'

# Thiết lập các tham số cấu hình chung
train_ratio = 0.8
valid_ratio = 0.1
img_size = 640

print(f"Thư mục nguồn dữ liệu: {src_folder}")
print(f"Thư mục đích lưu trữ (YOLO format): {dst_folder}")


# 1. Kiểm tra dữ liệu gốc
Liệt kê các tệp tin trong thư mục tải về để chuẩn bị đường dẫn.

In [ ]:
import os

if os.path.exists(src_folder):
    print('Contents of the source folder (src_folder):')
    print(os.listdir(src_folder))
    for sub in ['images', 'labels']:
        d = os.path.join(src_folder, sub)
        files = os.listdir(d) if os.path.isdir(d) else []
        print(f"  {sub}/: {len(files)} file, ví dụ: {files[:3]}")
else:
    print('Error: src_folder path does not exist!')


# 2. Định nghĩa cấu hình Tăng cường dữ liệu (Augmentation) và chia dữ liệu
Thiết lập tỷ lệ phân chia dữ liệu (Train 80%, Val 10%, Test 10%) và sử dụng thư viện `Albumentations` để tăng cường ảnh cho tập Train (giúp mô hình học tốt hơn dưới các góc độ, độ sáng khác nhau).

In [ ]:
import os
import glob
import shutil
import random
import cv2
import albumentations as A
from tqdm import tqdm

# 1. Khởi tạo cấu trúc các thư mục cần thiết theo chuẩn YOLO
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(dst_folder, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dst_folder, split, 'labels'), exist_ok=True)

# 2. Thu thập toàn bộ danh sách ảnh từ thư mục dữ liệu gốc
src_img_dir = os.path.join(src_folder, 'images')
src_lbl_dir = os.path.join(src_folder, 'labels')

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
all_images = [
    os.path.join(src_img_dir, f) for f in os.listdir(src_img_dir)
    if f.lower().endswith(IMG_EXTS)
]
assert len(all_images) > 0, f"Không tìm thấy ảnh nào trong {src_img_dir}"
print(f"Tìm thấy {len(all_images)} ảnh trong {src_img_dir}")
random.seed(42)
random.shuffle(all_images)

# 3. Tính toán kích thước phân bổ
total_count = len(all_images)
train_size = int(total_count * train_ratio)
valid_size = int(total_count * valid_ratio)

train_images = all_images[:train_size]
valid_images = all_images[train_size:train_size + valid_size]
test_images = all_images[train_size + valid_size:]

# 4. Định nghĩa cấu hình Augmentation bằng Albumentations
augmentations = A.Compose([
    A.Resize(img_size, img_size),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.2),
], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels']
))

# 5. Hàm tiến hành xử lý ảnh và copy nhãn
def process_and_split(image_list, split_name, is_train=False):
    print(f"Đang xử lý tập {split_name} (Tổng số: {len(image_list)})... ")
    for img_path in tqdm(image_list):
        filename = os.path.basename(img_path)
        label_filename = os.path.splitext(filename)[0] + '.txt'

        src_lbl_path = os.path.join(src_lbl_dir, label_filename)
        dst_img_path = os.path.join(dst_folder, split_name, 'images', filename)
        dst_lbl_path = os.path.join(dst_folder, split_name, 'labels', label_filename)

        # Sao chép file nhãn gốc nếu tồn tại
        if os.path.exists(src_lbl_path):
            shutil.copy(src_lbl_path, dst_lbl_path)
        else:
            # Tạo file nhãn rỗng
            with open(dst_lbl_path, 'w') as f:
                pass

        img = cv2.imread(img_path)
        if img is None:
            continue

        # Áp dụng tăng cường dữ liệu cho tập huấn luyện (Train)
        if is_train:
            bboxes = []
            class_labels = []
            if os.path.exists(src_lbl_path):
                with open(src_lbl_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            bboxes.append([float(x) for x in parts[1:]])
                            class_labels.append(int(float(parts[0])))
            try:
                transformed = augmentations(image=img, bboxes=bboxes, class_labels=class_labels)
                img_aug = transformed['image']
                bboxes_aug = transformed['bboxes']
                class_labels_aug = transformed['class_labels']

                cv2.imwrite(dst_img_path, img_aug)
                with open(dst_lbl_path, 'w') as f:
                    for cls_id, bbox in zip(class_labels_aug, bboxes_aug):
                        f.write(f"{cls_id} {' '.join(map(str, bbox))}\n")
            except Exception:
                # Dự phòng nếu lỗi bounding box
                img_resized = cv2.resize(img, (img_size, img_size))
                cv2.imwrite(dst_img_path, img_resized)
        else:
            # Tập Valid/Test chỉ resize giữ nguyên chất lượng gốc
            img_resized = cv2.resize(img, (img_size, img_size))
            cv2.imwrite(dst_img_path, img_resized)

# Tiến hành phân phối dữ liệu
process_and_split(train_images, 'train', is_train=True)
process_and_split(valid_images, 'valid', is_train=False)
process_and_split(test_images, 'test', is_train=False)

print("Hoàn thành toàn bộ quá trình chia dữ liệu và Augmentation!")

# 3. Vẽ kiểm tra trực quan ảnh huấn luyện đã xử lý
Kiểm tra các khung hình (bounding box) sau khi biến đổi ảnh (Augmentation) có còn khớp chính xác với biển báo hay không trước khi đưa vào huấn luyện.

*(Hiển thị ngẫu nhiên 3 ảnh mẫu cùng nhãn dán tương ứng trong tập train)*

In [ ]:
import os
import cv2
import random
import matplotlib.pyplot as plt
import numpy as np
import albumentations as A
from tqdm import tqdm

# Tải danh sách lớp (classes) từ file classes_en.txt hoặc classes_vie.txt trong src_folder
classes = []
classes_file_path = os.path.join(src_folder, 'classes_en.txt')
if not os.path.exists(classes_file_path):
    classes_file_path = os.path.join(src_folder, 'classes.txt')

if os.path.exists(classes_file_path):
    with open(classes_file_path, 'r', encoding='utf-8') as f:
        classes = [line.strip() for line in f.readlines() if line.strip()]
else:
    # Fallback nếu không tìm thấy file
    classes = [str(i) for i in range(100)]

print(f"Loaded {len(classes)} classes successfully.")

# Định nghĩa augmentations
augmentations = A.Compose([
    A.Resize(img_size, img_size),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.Affine(shear=5, p=0.3),
    A.GaussianBlur(blur_limit=(3,5), p=0.2),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.HueSaturationValue(p=0.3),
], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels']
))

# Hàm vẽ bounding box để kiểm tra trực quan
def draw_boxes(image_path, label_path, class_names):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image.shape

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    # Sửa lỗi ép kiểu float-string sang int
                    cls_id = int(float(parts[0]))
                    x_c, y_c, bw, bh = map(float, parts[1:])

                    # Chuyển đổi từ tọa độ YOLO sang tọa độ pixel
                    x1 = int((x_c - bw / 2) * w)
                    y1 = int((y_c - bh / 2) * h)
                    x2 = int((x_c + bw / 2) * w)
                    y2 = int((y_c + bh / 2) * h)

                    # Vẽ hình chữ nhật và viết tên lớp lên ảnh
                    cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    cls_name = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                    cv2.putText(image, cls_name, (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return image

# Lấy ngẫu nhiên 3 ảnh đã được xử lý (augmented) từ tập train để hiển thị
train_img_dir = os.path.join(dst_folder, 'train/images')
train_imgs = [f for f in os.listdir(train_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]

if len(train_imgs) == 0:
    print("Cảnh báo: Không tìm thấy ảnh huấn luyện nào trong thư mục đích.")
else:
    sample_imgs = random.sample(train_imgs, min(3, len(train_imgs)))
    fig, axes = plt.subplots(1, len(sample_imgs), figsize=(18, 6))
    if len(sample_imgs) == 1:
        axes = [axes]

    for i, img_name in enumerate(sample_imgs):
        img_path = os.path.join(train_img_dir, img_name)
        lbl_path = os.path.join(dst_folder, 'train/labels', os.path.splitext(img_name)[0] + '.txt')
        vis_img = draw_boxes(img_path, lbl_path, classes)
        axes[i].imshow(vis_img)
        axes[i].axis('off')
        axes[i].set_title(f"Augmented: {img_name}")

    plt.tight_layout()
    plt.show()

---

# 4. Tạo file cấu hình dataset.yaml cho YOLO
Tạo tệp YAML định nghĩa các đường dẫn ảnh Train/Val/Test và danh sách tên các loại biển báo để truyền vào mô hình YOLO.

In [ ]:
import yaml

# Định nghĩa cấu hình dataset
yolo_config = {
    'path': dst_folder,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(classes),
    'names': {i: name for i, name in enumerate(classes)}
}

yaml_path = os.path.join(dst_folder, 'dataset.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(yolo_config, f, default_flow_style=False, allow_unicode=True)

print(f"Đã tạo thành công file cấu hình YOLO tại: {yaml_path}")
with open(yaml_path, 'r', encoding='utf-8') as f:
    print(f.read())

# 5. Cài đặt thư viện và huấn luyện mô hình YOLO11N
Tải mô hình pre-trained YOLO11N và tiến hành huấn luyện (fine-tune) trên tập dữ liệu biển báo.

In [ ]:
!pip install -q ultralytics

import torch
from ultralytics import YOLO

# Kiểm tra thiết bị phần cứng khả dụng (GPU / CPU)
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Khởi tạo mô hình YOLO11N
model = YOLO('yolo11n.pt')

# Tiến hành huấn luyện mô hình
results = model.train(
    data=yaml_path,
    epochs=10,             # Số epoch tối ưu hóa chạy nhanh gọn để trình diễn và kiểm thử
    imgsz=img_size,        # Kích thước ảnh đầu vào (640x640)
    batch=16,              # Batch size cân bằng bộ nhớ RAM/VRAM GPU
    device=device,
    workers=4,
    project=project_dir,    
    name='yolo11n_run',
    exist_ok=True
)

### 6. Đánh giá chất lượng và kiểm thử trực quan trên tập Test độc lập
Đánh giá định lượng (tính toán chính xác các chỉ số mAP50, Precision, Recall) trên tập dữ liệu kiểm thử độc lập mà mô hình chưa từng thấy trong quá trình học.

In [ ]:
import os
import glob
import random
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO

run_dir = os.path.join(project_dir, 'yolo11n_run')
best_model_path = os.path.join(run_dir, 'weights', 'best.pt')

if os.path.exists(best_model_path):
    trained_model = YOLO(best_model_path)

    print("--- Đang tiến hành đánh giá trên tập kiểm thử (Test Set) ---")
    metrics = trained_model.val(data=yaml_path, split='test', verbose=False)

    print(f"Kết quả đánh giá tập Test:")
    print(f"- mAP50: {metrics.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"- mAP50-95: {metrics.results_dict['metrics/mAP50-95(B)']:.4f}")
    print(f"- Precision (B): {metrics.results_dict['metrics/precision(B)']:.4f}")
    print(f"- Recall (B): {metrics.results_dict['metrics/recall(B)']:.4f}")
    print("---------------------------------------------------")

    test_img_dir = os.path.join(dst_folder, 'test/images')
    test_images = [p for ext in ('jpg', 'jpeg', 'png') for p in glob.glob(os.path.join(test_img_dir, f'*.{ext}'))]

    if len(test_images) > 0:
        sample_test_images = random.sample(test_images, min(3, len(test_images)))
        fig, axes = plt.subplots(1, len(sample_test_images), figsize=(18, 6))
        if len(sample_test_images) == 1:
            axes = [axes]

        for i, img_path in enumerate(sample_test_images):
            results_pred = trained_model.predict(source=img_path, conf=0.25, verbose=False)
            annotated_img = results_pred[0].plot()
            annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

            axes[i].imshow(annotated_img)
            axes[i].axis('off')
            axes[i].set_title(f"Dự đoán: {os.path.basename(img_path)}")
        plt.tight_layout()
        plt.show()
    else:
        print("Không tìm thấy hình ảnh nào trong thư mục kiểm thử.")
else:
    print(f"Không tìm thấy trọng số tốt nhất tại: {best_model_path}")

# 7. Trực quan hóa kết quả đồ thị và dự đoán ảnh thực tế
Hiển thị đồ thị diễn biến các chỉ số Loss/mAP qua từng epoch, đồng thời lấy ngẫu nhiên một số bức ảnh từ tập kiểm thử để mô hình dự đoán và vẽ trực tiếp kết quả lên màn hình.

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO

run_dir = os.path.join(project_dir, 'yolo11n_run')
results_png = os.path.join(run_dir, 'results.png')

if os.path.exists(results_png):
    plt.figure(figsize=(12, 10))
    img_results = cv2.imread(results_png)
    img_results = cv2.cvtColor(img_results, cv2.COLOR_BGR2RGB)
    plt.imshow(img_results)
    plt.axis('off')
    plt.title("Đồ thị hiệu năng huấn luyện YOLO11n (Loss & mAP Metrics)", fontsize=16)
    plt.show()
else:
    print("Không tìm thấy file kết quả đồ thị huấn luyện results.png.")

best_model_path = os.path.join(run_dir, 'weights', 'best.pt')
if os.path.exists(best_model_path):
    trained_model = YOLO(best_model_path)

    test_img_dir = os.path.join(dst_folder, 'test/images')
    test_images = [p for ext in ('jpg', 'jpeg', 'png') for p in glob.glob(os.path.join(test_img_dir, f'*.{ext}'))]

    if len(test_images) > 0:
        sample_test_images = random.sample(test_images, min(3, len(test_images)))

        fig, axes = plt.subplots(1, len(sample_test_images), figsize=(18, 6))
        if len(sample_test_images) == 1:
            axes = [axes]

        for i, img_path in enumerate(sample_test_images):
            results_pred = trained_model.predict(source=img_path, conf=0.25, verbose=False)

            annotated_img = results_pred[0].plot()
            annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

            axes[i].imshow(annotated_img)
            axes[i].axis('off')
            axes[i].set_title(f"Dự đoán: {os.path.basename(img_path)}")

        plt.tight_layout()
        plt.show()
    else:
        print("Không tìm thấy hình ảnh nào trong thư mục kiểm thử 'test/images'.")
else:
    print(f"Không tìm thấy trọng số tốt nhất tại đường dẫn: {best_model_path}")